# TP-1: CleaningData - Applying OOP and Design Patterns

This notebook builds **one cumulative CSV-cleaning tool** for `dataset/sample_data.csv`.
Every exercise adds one building block, and Exercise 5 combines everything into a single
pipeline that loads, cleans, transforms and saves the data to `output/cleaned_data.csv`.

How the 5 exercises connect:

| # | Exercise | What it adds | Role in the final pipeline |
|---|----------|--------------|----------------------------|
| 1 | CSVReader (OOP foundation) | reads and previews the CSV | `load()` step |
| 2 | Strategy pattern (missing values) | interchangeable ways of handling NaN | `clean()` step |
| 3 | Decorators (logging + timing) | reusable logging/timing around methods | wraps `read`, `clean`, `run` |
| 4 | Factory pattern (transformations) | transformation objects created from a string | `transform()` step |
| 5 | Template Method (full pipeline) | fixed `load -> clean -> transform -> save` workflow | `run()` |

**Table of contents:** Exercise 1 -> Exercise 2 -> Exercise 3 -> Exercise 4 -> Exercise 5 -> Conclusion.

## Exercise 1: Build the CSVReader Class (OOP Foundation)

### Objective
- Reinforce OOP fundamentals: classes, constructors, methods, encapsulation.
- Learn to design reusable data classes for DS projects.

### Concept
A CSVReader class should encapsulate all functionality related to reading CSV files.

### Instructions
- Create a class `CSVReader`.
- Define an `__init__` method that takes `file_path`.
- Add a `read()` method that loads the CSV file using pandas.
- Add a `preview(n)` method that prints the top n rows.

### How my solution meets this
- `CSVReader.__init__(file_path)` keeps the path in the private `_file_path` and the data in `_df` (encapsulation), exposed through the read-only properties `file_path` and `df`.
- `read()` loads the CSV with `pd.read_csv` and raises `FileNotFoundError` for a missing file; `preview(n)` prints (and returns) the top n rows and raises `RuntimeError` if called before `read()`.

In [1]:
import contextlib
import functools
import io
import os
import time
from abc import ABC, abstractmethod

import pandas as pd


class CSVReader:
    """Reads a CSV file and gives a quick preview of its contents."""

    def __init__(self, file_path: str) -> None:
        self._file_path = file_path
        self._df: pd.DataFrame | None = None

    @property
    def file_path(self) -> str:
        """Read-only access to the file path."""
        return self._file_path

    @property
    def df(self) -> pd.DataFrame | None:
        """Read-only access to the loaded DataFrame (None until read())."""
        return self._df

    def read(self) -> pd.DataFrame:
        """Load the CSV file into the internal DataFrame and return it."""
        if not os.path.exists(self._file_path):
            raise FileNotFoundError(f"CSV file not found: {self._file_path}")
        self._df = pd.read_csv(self._file_path)
        return self._df

    def preview(self, n: int = 5) -> pd.DataFrame:
        """Print the top n rows of the loaded DataFrame and return them."""
        if self._df is None:
            raise RuntimeError("No data loaded yet: call read() before preview().")
        top = self._df.head(n)
        print(top)
        return top

**Demo:** we load `dataset/sample_data.csv` with our class and print its first 4 rows.

In [4]:
reader = CSVReader("../dataset/sample_data.csv")
df = reader.read()

print("file:", reader.file_path)
print("shape:", df.shape)
print("\npreview(4):")
reader.preview(4)

file: ../dataset/sample_data.csv
shape: (7, 7)

preview(4):
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0


**Checks:** the reader returns a non-empty DataFrame, `preview(n)` prints exactly the rows it returns, and clear errors are raised for a missing file and for previewing before reading.

In [8]:
test_reader = CSVReader("../dataset/sample_data.csv")
test_df = test_reader.read()

assert isinstance(test_df, pd.DataFrame) and not test_df.empty
assert test_reader.file_path == "../dataset/sample_data.csv"
assert test_reader.df is test_df  # read-only property exposes what read() loaded

# preview prints exactly the rows it returns
buffer = io.StringIO()
with contextlib.redirect_stdout(buffer):
    printed = test_reader.preview(3)
assert len(printed) == 3 and str(printed) in buffer.getvalue()

# missing file -> clear error
try:
    CSVReader("dataset/does_not_exist.csv").read()
    raise AssertionError("FileNotFoundError was not raised")
except FileNotFoundError as error:
    print("OK missing file ->", error)

# preview before read -> clear error
try:
    CSVReader("dataset/sample_data.csv").preview(2)
    raise AssertionError("RuntimeError was not raised")
except RuntimeError as error:
    print("OK preview before read ->", error)

print("Exercise 1: all checks passed.")

OK missing file -> CSV file not found: dataset/does_not_exist.csv
OK preview before read -> No data loaded yet: call read() before preview().
Exercise 1: all checks passed.


## Exercise 2: Apply Pattern for Missing Values Handling

### Objective
- Learn the Strategy Pattern - one interface, many interchangeable behaviors.
- Practice inheritance and polymorphism.

### Concept
We often need different strategies to handle missing data (drop, fill mean, fill mode, etc.).
The pattern allows flexible switching between methods without changing the main code.

### Instructions
- Create an abstract class `MissingValueStrategy` with a method `handle(df)`.
- Create subclasses:
  - `DropMissing`
  - `FillMean`
  - `FillMode`
- Create a `DataCleaner` class that applies the chosen strategy.

### Design decisions (where the TP is ambiguous)
- `DropMissing`: drops every row that contains at least one missing value.
- `FillMean`: fills **numeric columns only** with their mean; non-numeric columns are left untouched.
- `FillMode`: fills each column with its first mode (first value when several share the max count); columns that are entirely empty are skipped.
- Every strategy works on a **copy** of the DataFrame: the caller's DataFrame is never modified.

### How my solution meets this
- `MissingValueStrategy` is the abstract interface (`handle(df)` declared with `@abstractmethod`); `DropMissing`, `FillMean` and `FillMode` inherit from it, each giving `handle()` its own behaviour.
- `DataCleaner` receives **any** strategy in its constructor and calls `strategy.handle(df)` inside `clean()`, so the strategy is swapped without touching `DataCleaner` itself (polymorphism).

In [9]:
class MissingValueStrategy(ABC):
    """Strategy interface: how to handle missing values in a DataFrame."""

    @abstractmethod
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Return a new DataFrame with the missing values handled."""


class DropMissing(MissingValueStrategy):
    """Drop every row that contains at least one missing value."""

    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.dropna()


class FillMean(MissingValueStrategy):
    """Fill numeric columns with their mean; leave other columns untouched."""

    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        filled = df.copy()  # never modify the caller's DataFrame
        numeric_cols = filled.select_dtypes(include="number").columns
        filled[numeric_cols] = filled[numeric_cols].fillna(filled[numeric_cols].mean())
        return filled


class FillMode(MissingValueStrategy):
    """Fill each column with its first mode; skip columns that are all empty."""

    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        filled = df.copy()  # never modify the caller's DataFrame
        for column in filled.columns:
            modes = filled[column].mode()  # one mode per column: needs a small loop
            if not modes.empty:  # an all-NaN column has no mode: leave it as is
                filled[column] = filled[column].fillna(modes.iloc[0])
        return filled


class DataCleaner:
    """Applies a MissingValueStrategy to a DataFrame (Strategy Pattern)."""

    def __init__(self, strategy: MissingValueStrategy) -> None:
        if not isinstance(strategy, MissingValueStrategy):
            raise TypeError("strategy must be a MissingValueStrategy instance")
        self._strategy = strategy

    @property
    def strategy(self) -> MissingValueStrategy:
        """Read-only access to the current strategy."""
        return self._strategy

    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        """Delegate missing-value handling to the current strategy."""
        return self._strategy.handle(df)

**Demo:** the three strategies run through the *same* `DataCleaner` on the raw data; we compare shape and number of missing values before/after each one.

In [10]:
original = df.copy(deep=True)  # reference copy: lets us prove nothing mutates it
print(f"raw data: shape={df.shape}, missing={int(df.isna().sum().sum())}\n")

for strategy in (DropMissing(), FillMean(), FillMode()):
    result = DataCleaner(strategy).clean(df)
    print(f"{type(strategy).__name__:12s} -> shape={result.shape}, missing={int(result.isna().sum().sum())}")

print("\nFillMean result (means filled in):")
DataCleaner(FillMean()).clean(df)

raw data: shape=(7, 7), missing=7

DropMissing  -> shape=(2, 7), missing=0
FillMean     -> shape=(7, 7), missing=0
FillMode     -> shape=(7, 7), missing=0

FillMean result (means filled in):


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.00,165.0,68.0,New York,85.0
1,2,Bob,30.25,172.0,70.0,Los Angeles,90.0
2,3,Charlie,35.00,168.0,72.0,Chicago,86.0
3,4,David,30.25,167.0,80.0,Houston,75.0
4,5,Eva,27.00,160.0,55.0,New York,88.0
5,6,Frank,30.00,175.0,85.0,Los Angeles,86.0
6,7,Grace,30.25,162.0,60.0,Chicago,92.0


**Checks:** each strategy behaves as specified, all three are interchangeable through `DataCleaner` without changing its code, a wrong strategy type is rejected, and the original DataFrame stays untouched.

In [11]:
# DropMissing leaves no missing value at all
dropped = DataCleaner(DropMissing()).clean(df)
assert int(dropped.isna().sum().sum()) == 0
assert len(dropped) < len(df)  # rows with NaN were removed

# FillMean leaves no NaN in numeric columns; non-numeric columns are untouched
mean_filled = DataCleaner(FillMean()).clean(df)
numeric_cols = df.select_dtypes(include="number").columns
non_numeric_cols = df.select_dtypes(exclude="number").columns
assert int(mean_filled[numeric_cols].isna().sum().sum()) == 0
pd.testing.assert_frame_equal(mean_filled[non_numeric_cols], df[non_numeric_cols])
assert int(mean_filled.isna().sum().sum()) == 0  # this dataset has no NaN left anywhere

# FillMode leaves no NaN in every column that has a mode
mode_filled = DataCleaner(FillMode()).clean(df)
for column in df.columns:
    if not df[column].mode().empty:
        assert mode_filled[column].isna().sum() == 0

# Polymorphism: the same DataCleaner code with three different behaviours
for strategy in (DropMissing(), FillMean(), FillMode()):
    output = DataCleaner(strategy).clean(df)
    assert isinstance(output, pd.DataFrame)

# A non-strategy object is rejected
try:
    DataCleaner("mean")
    raise AssertionError("TypeError was not raised")
except TypeError as error:
    print("OK wrong strategy type ->", error)

# The caller's DataFrame is never mutated
pd.testing.assert_frame_equal(df, original)
print("Exercise 2: all checks passed.")

OK wrong strategy type -> strategy must be a MissingValueStrategy instance
Exercise 2: all checks passed.


## Exercise 3: Add Decorators for Logging and Timing

### Objective
- Learn the Decorator Pattern to add reusable functionality (logging, timing).
- Understand how decorators support the "Open/Closed Principle".

### Concept
Decorators wrap functions to extend their behavior without altering the original code.

### Instructions
- Create two decorators:
  - `@log_action` -> logs when a method starts and finishes.
  - `@log_time` -> calculates and prints execution time.
- Apply them to methods in `CSVReader` or `DataCleaner`.

### Design decisions (where the TP is ambiguous)
- Both decorators use `functools.wraps`, so the wrapped method keeps its original name and docstring.
- `@log_action` prints `[LOG] <name> started` and `[LOG] <name> finished`; `@log_time` measures the call with `time.perf_counter()` and prints `[TIME] <name> took X.XXXXs`.
- They are applied to `CSVReader.read()` and `DataCleaner.clean()` by re-declaring those two classes below: same classes as Exercises 1-2, only decorated - behaviour is extended without modifying the original code (Open/Closed Principle). All later exercises use these decorated versions.

### How my solution meets this
- `log_action` and `log_time` are two small reusable decorators that add logging/timing around any function without changing its body.
- The decorated `CSVReader.read()` and `DataCleaner.clean()` apply them to methods from Exercises 1-2, and their printed output is visible in the demo.

In [12]:
def log_action(func):
    """Log when the wrapped method starts and finishes."""

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[LOG] {func.__name__} started")
        result = func(*args, **kwargs)
        print(f"[LOG] {func.__name__} finished")
        return result

    return wrapper


def log_time(func):
    """Print the execution time of the wrapped method."""

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"[TIME] {func.__name__} took {time.perf_counter() - start:.4f}s")
        return result

    return wrapper

**Updated versions** - same classes as Exercises 1-2, now decorated. Nothing else changed: the decorators add behaviour while the original method bodies stay untouched (Open/Closed Principle). Every cell below this one uses these versions.

In [13]:
class CSVReader:
    """Reads a CSV file and gives a quick preview of its contents."""

    def __init__(self, file_path: str) -> None:
        self._file_path = file_path
        self._df: pd.DataFrame | None = None

    @property
    def file_path(self) -> str:
        """Read-only access to the file path."""
        return self._file_path

    @property
    def df(self) -> pd.DataFrame | None:
        """Read-only access to the loaded DataFrame (None until read())."""
        return self._df

    @log_action
    @log_time
    def read(self) -> pd.DataFrame:
        """Load the CSV file into the internal DataFrame and return it."""
        if not os.path.exists(self._file_path):
            raise FileNotFoundError(f"CSV file not found: {self._file_path}")
        self._df = pd.read_csv(self._file_path)
        return self._df

    def preview(self, n: int = 5) -> pd.DataFrame:
        """Print the top n rows of the loaded DataFrame and return them."""
        if self._df is None:
            raise RuntimeError("No data loaded yet: call read() before preview().")
        top = self._df.head(n)
        print(top)
        return top


class DataCleaner:
    """Applies a MissingValueStrategy to a DataFrame (Strategy Pattern)."""

    def __init__(self, strategy: MissingValueStrategy) -> None:
        if not isinstance(strategy, MissingValueStrategy):
            raise TypeError("strategy must be a MissingValueStrategy instance")
        self._strategy = strategy

    @property
    def strategy(self) -> MissingValueStrategy:
        """Read-only access to the current strategy."""
        return self._strategy

    @log_action
    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        """Delegate missing-value handling to the current strategy."""
        return self._strategy.handle(df)

**Demo:** the decorated methods now print their start/finish log and their execution time, while the behaviour of the classes stays exactly the same.

In [15]:
decorated_reader = CSVReader("../dataset/sample_data.csv")
df = decorated_reader.read()  # prints log + timing

print("\npreview(3):")
decorated_reader.preview(3)

print("\nclean with FillMean:")
df_clean = DataCleaner(FillMean()).clean(df)  # prints log
print("missing after clean:", int(df_clean.isna().sum().sum()))

[LOG] read started
[TIME] read took 0.0039s
[LOG] read finished

preview(3):
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN

clean with FillMean:
[LOG] clean started
[LOG] clean finished
missing after clean: 0


**Checks:** the decorators really print log/timing lines, `functools.wraps` preserves the function name and docstring, and the decorated classes return the same results as before.

In [17]:
# functools.wraps keeps name and docstring
assert CSVReader.read.__name__ == "read"
assert CSVReader.read.__doc__ and "Load the CSV file" in CSVReader.read.__doc__
assert DataCleaner.clean.__name__ == "clean"

# the decorators themselves preserve name and docstring on any function
def sample_method():
    """Sample docstring."""

wrapped = log_action(log_time(sample_method))
assert wrapped.__name__ == "sample_method"
assert wrapped.__doc__ == "Sample docstring."

# the decorators really print their lines
buffer = io.StringIO()
with contextlib.redirect_stdout(buffer):
    CSVReader("../dataset/sample_data.csv").read()
    DataCleaner(FillMean()).clean(df)
logs = buffer.getvalue()
assert "[LOG] read started" in logs and "[LOG] read finished" in logs
assert "[TIME] read took" in logs
assert "[LOG] clean started" in logs and "[LOG] clean finished" in logs

# behaviour unchanged: same data, still cleaned correctly
assert int(DataCleaner(DropMissing()).clean(df).isna().sum().sum()) == 0
pd.testing.assert_frame_equal(df, original)
print("Exercise 3: all checks passed.")

[LOG] clean started
[LOG] clean finished
Exercise 3: all checks passed.


## Exercise 4: Implement Factory Pattern for Data Transformations

### Objective
- Practice Factory Pattern for scalable creation of transformation objects.
- Apply abstraction and composition.

### Concept
Instead of manually creating objects, use a factory that decides what transformation to apply.

### Instructions
- Create an abstract class `DataTransform` with a method `apply(df)`.
- Implement subclasses:
  - `NormalizeColumns`
  - `RemoveDuplicates`
  - `StandardizeText`
- Create a `TransformFactory` that returns transformation objects based on string input.

### Design decisions (where the TP is ambiguous)
- `NormalizeColumns`: interpretation - it normalises the column **names** (strip whitespace, lowercase, spaces -> underscores), not the values.
- `StandardizeText`: strips whitespace and lowercases the text (object/string) columns only.
- `RemoveDuplicates`: drops exact duplicate rows and resets the index.
- `TransformFactory`: a dictionary registry maps a string key (`"normalize_columns"`, `"remove_duplicates"`, `"standardize_text"`) to a class; keys are matched case-insensitively and an unknown key raises `ValueError` listing the valid names.

### How my solution meets this
- `DataTransform` is the abstract interface (`apply(df)`); `NormalizeColumns`, `RemoveDuplicates` and `StandardizeText` each implement exactly one transformation.
- `TransformFactory.create(name)` builds the right object from the registry, so the pipeline only passes strings around, and a new transform is added by registering one more entry (scalable, no `if/elif` chain to grow).

In [18]:
class DataTransform(ABC):
    """Abstract interface for one DataFrame transformation."""

    @abstractmethod
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        """Return a new transformed DataFrame."""


class NormalizeColumns(DataTransform):
    """Strip, lowercase and underscore the column names."""

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        normalized = df.copy()  # never modify the caller's DataFrame
        normalized.columns = (
            normalized.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)
        )
        return normalized


class RemoveDuplicates(DataTransform):
    """Drop exact duplicate rows and reset the index."""

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.drop_duplicates().reset_index(drop=True)


class StandardizeText(DataTransform):
    """Strip whitespace and lowercase the text columns."""

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        standardized = df.copy()  # never modify the caller's DataFrame
        text_cols = standardized.select_dtypes(include=["object", "string"]).columns
        standardized[text_cols] = standardized[text_cols].apply(
            lambda column: column.str.strip().str.lower()
        )
        return standardized


class TransformFactory:
    """Creates DataTransform objects from a string name (Factory Pattern)."""

    _REGISTRY: dict[str, type[DataTransform]] = {
        "normalize_columns": NormalizeColumns,
        "remove_duplicates": RemoveDuplicates,
        "standardize_text": StandardizeText,
    }

    @classmethod
    def create(cls, name: str) -> DataTransform:
        """Return the transform registered under `name` (case-insensitive)."""
        key = name.strip().lower()
        if key not in cls._REGISTRY:
            valid = ", ".join(sorted(cls._REGISTRY))
            raise ValueError(f"Unknown transform {name!r}. Valid names: {valid}")
        return cls._REGISTRY[key]()

**Demo:** `StandardizeText` runs on the real CSV (names and cities become lowercase, numbers untouched). The provided CSV already has clean column names and no duplicate rows, so `NormalizeColumns` and `RemoveDuplicates` are demonstrated on a small constructed "messy" DataFrame where their effect is visible.

In [19]:
# StandardizeText on the real CSV
print("before:", df["name"].head(3).tolist(), "/", df["city"].head(3).tolist())
standardized = TransformFactory.create("standardize_text").apply(df)
print("after: ", standardized["name"].head(3).tolist(), "/", standardized["city"].head(3).tolist())

# NormalizeColumns + RemoveDuplicates on a small messy frame
messy = pd.DataFrame(
    {
        " First Name ": [" Alice ", "Bob", "Bob "],
        "LAST NAME": ["SMITH", "doe", "doe"],
        "City": ["NEW YORK", "los angeles", "los angeles "],
    }
)
print("\nmessy columns:   ", list(messy.columns))
normalized = TransformFactory.create("normalize_columns").apply(messy)
print("normalized names:", list(normalized.columns))

standardized_messy = TransformFactory.create("standardize_text").apply(messy)
print("\nafter standardize_text:")
print(standardized_messy)
deduped = TransformFactory.create("remove_duplicates").apply(standardized_messy)
print(f"\nafter remove_duplicates -> {len(deduped)} rows (was {len(standardized_messy)})")
print(deduped)

# the factory is case-insensitive
print("\nfactory('Remove_Duplicates') ->", type(TransformFactory.create("Remove_Duplicates")).__name__)

before: ['Alice', 'Bob', 'Charlie'] / ['New York', 'Los Angeles', 'Chicago']
after:  ['alice', 'bob', 'charlie'] / ['new york', 'los angeles', 'chicago']

messy columns:    [' First Name ', 'LAST NAME', 'City']
normalized names: ['first_name', 'last_name', 'city']

after standardize_text:
   First Name  LAST NAME         City
0        alice     smith     new york
1          bob       doe  los angeles
2          bob       doe  los angeles

after remove_duplicates -> 2 rows (was 3)
   First Name  LAST NAME         City
0        alice     smith     new york
1          bob       doe  los angeles

factory('Remove_Duplicates') -> RemoveDuplicates


**Checks:** each transform does what it promises, the factory is case-insensitive and rejects unknown names with a clear `ValueError`, and the caller's DataFrame is never mutated.

In [20]:
# NormalizeColumns -> clean column names
norm = TransformFactory.create("normalize_columns").apply(messy)
assert list(norm.columns) == ["first_name", "last_name", "city"]

# StandardizeText -> stripped/lowercased text, numeric columns untouched
std = TransformFactory.create("standardize_text").apply(df)
assert std["name"].str.islower().all() and std["city"].str.islower().all()
pd.testing.assert_series_equal(std["age"], df["age"])

# RemoveDuplicates -> no duplicate rows left, index reset
deduped = TransformFactory.create("remove_duplicates").apply(
    TransformFactory.create("standardize_text").apply(messy)
)
assert int(deduped.duplicated().sum()) == 0
assert list(deduped.index) == [0, 1]  # 3 messy rows -> 2 unique rows

# Factory: case-insensitive names, clear ValueError otherwise
assert isinstance(TransformFactory.create("Remove_Duplicates"), RemoveDuplicates)
try:
    TransformFactory.create("scale_values")
    raise AssertionError("ValueError was not raised")
except ValueError as error:
    print("OK unknown transform ->", error)

# The caller's DataFrame is never mutated
pd.testing.assert_frame_equal(df, original)
print("Exercise 4: all checks passed.")

OK unknown transform -> Unknown transform 'scale_values'. Valid names: normalize_columns, remove_duplicates, standardize_text
Exercise 4: all checks passed.


## Exercise 5: Build a Full Cleaning Pipeline (Template Method Pattern)

### Objective
- Combine all previous concepts into one cleaning pipeline.
- Use the Template Method Pattern to define a consistent workflow.

### Concept
The Template Method defines the skeleton of a process and lets subclasses override specific steps.

### Instructions
- Create an abstract class `DataPipeline` with a `run()` method defining steps:
  - `load()`
  - `clean()`
  - `transform()`
  - `save()`
- Create `CSVDataPipeline` that implements each step using previous classes.

### Design decisions (where the TP is ambiguous)
- `run()` is the template method: it calls `load() -> clean() -> transform() -> save()` in that fixed order (itself decorated with `@log_action` and `@log_time`) and returns the final DataFrame.
- `CSVDataPipeline(input_path, output_path, strategy, transform_names)` composes the classes of Exercises 1-4: `CSVReader` for `load()`, `DataCleaner` with the given `MissingValueStrategy` for `clean()`, and one `TransformFactory.create(name)` per entry of `transform_names` for `transform()`.
- `save()` creates the output folder if it is missing and writes the result with `to_csv(..., index=False)` to `output/cleaned_data.csv`.

### How my solution meets this
- `DataPipeline.run()` fixes the 4-step workflow while `load/clean/transform/save` stay abstract; `CSVDataPipeline` implements each step with the earlier classes - workflow and concrete steps are separated (Template Method).
- Changing the strategy or the list of transform names changes the pipeline's behaviour through constructor arguments only: neither `run()` nor the step code needs to change.

In [21]:
class DataPipeline(ABC):
    """Template Method: fixed load -> clean -> transform -> save workflow."""

    def __init__(self, output_path: str) -> None:
        self._output_path = output_path
        self._df: pd.DataFrame | None = None

    @property
    def output_path(self) -> str:
        """Read-only access to where the pipeline saves its result."""
        return self._output_path

    @property
    def df(self) -> pd.DataFrame | None:
        """Read-only access to the DataFrame currently held by the pipeline."""
        return self._df

    @log_action
    @log_time
    def run(self) -> pd.DataFrame:
        """Execute the four steps in order and return the cleaned DataFrame."""
        self.load()
        self.clean()
        self.transform()
        self.save()
        return self._df

    @abstractmethod
    def load(self) -> None:
        """Read the raw data into the pipeline."""

    @abstractmethod
    def clean(self) -> None:
        """Handle the missing values."""

    @abstractmethod
    def transform(self) -> None:
        """Apply the configured transformations."""

    @abstractmethod
    def save(self) -> None:
        """Write the result to the output path."""


class CSVDataPipeline(DataPipeline):
    """CSV cleaning pipeline built from the classes of Exercises 1-4."""

    def __init__(
        self,
        input_path: str,
        output_path: str,
        strategy: MissingValueStrategy,
        transform_names: list[str],
    ) -> None:
        super().__init__(output_path)
        self._reader = CSVReader(input_path)  # Exercise 1 (+3)
        self._strategy = strategy  # Exercise 2
        self._transform_names = list(transform_names)  # Exercise 4

    def load(self) -> None:
        self._df = self._reader.read()

    def clean(self) -> None:
        self._df = DataCleaner(self._strategy).clean(self._df)  # Exercises 2 + 3

    def transform(self) -> None:
        for name in self._transform_names:
            self._df = TransformFactory.create(name).apply(self._df)  # Exercise 4

    def save(self) -> None:
        directory = os.path.dirname(self._output_path)
        if directory:
            os.makedirs(directory, exist_ok=True)  # create output/ if missing
        self._df.to_csv(self._output_path, index=False)

**Demo:** we run the full pipeline on `dataset/sample_data.csv` with the `FillMean` strategy and three transforms. `run()` prints the workflow log/timing, we show the cleaned result, and `output/cleaned_data.csv` is written.

In [23]:
pipeline = CSVDataPipeline(
    input_path="../dataset/sample_data.csv",
    output_path="output/cleaned_data.csv",
    strategy=FillMean(),
    transform_names=["normalize_columns", "standardize_text", "remove_duplicates"],
)

cleaned = pipeline.run()  # load -> clean -> transform -> save (logged + timed)

print("\nshape:", cleaned.shape,
      "| missing:", int(cleaned.isna().sum().sum()),
      "| duplicates:", int(cleaned.duplicated().sum()))
cleaned

[LOG] run started
[LOG] read started
[TIME] read took 0.0040s
[LOG] read finished
[LOG] clean started
[LOG] clean finished
[TIME] run took 0.0392s
[LOG] run finished

shape: (7, 7) | missing: 0 | duplicates: 0


,id,name,age,height_cm,weight_kg,city,score
0,1,alice,29.00,165.0,68.0,new york,85.0
1,2,bob,30.25,172.0,70.0,los angeles,90.0
2,3,charlie,35.00,168.0,72.0,chicago,86.0
3,4,david,30.25,167.0,80.0,houston,75.0
4,5,eva,27.00,160.0,55.0,new york,88.0
5,6,frank,30.00,175.0,85.0,los angeles,86.0
6,7,grace,30.25,162.0,60.0,chicago,92.0


**Checks:** the output file exists and can be re-read with no missing values and no duplicates, the template method returned exactly what was saved, the abstract pipeline cannot be instantiated, and the raw dataset was never modified.

In [24]:
# The file exists and can be re-read
assert os.path.exists("output/cleaned_data.csv")
reread = pd.read_csv("output/cleaned_data.csv")

# No missing values, no duplicates, no rows lost
assert int(reread.isna().sum().sum()) == 0
assert int(reread.duplicated().sum()) == 0
assert len(reread) == len(df)  # FillMean keeps every row

# The transforms were really applied: clean names, lowercase text
assert list(reread.columns) == ["id", "name", "age", "height_cm", "weight_kg", "city", "score"]
assert reread["name"].str.islower().all() and reread["city"].str.islower().all()

# The template method returned the same data it saved
pd.testing.assert_frame_equal(cleaned.reset_index(drop=True), reread)

# The abstract pipeline cannot be instantiated
try:
    DataPipeline("output/abstract.csv")
    raise AssertionError("TypeError was not raised")
except TypeError as error:
    print("OK abstract DataPipeline ->", error)

# The raw dataset was never modified
pd.testing.assert_frame_equal(df, original)
print("Exercise 5: all checks passed.")
print("Saved:", pipeline.output_path)

OK abstract DataPipeline -> Can't instantiate abstract class DataPipeline without an implementation for abstract methods 'clean', 'load', 'save', 'transform'
Exercise 5: all checks passed.
Saved: output/cleaned_data.csv
